# 🔥 ShareChat — Trending Tags System
### APM Assignment | Powered by Claude AI + Web Search

**What this notebook does:**
- Fetches **live, real-time trends** from the web using Claude's web search
- Ranks them for a Hindi-speaking Indian audience
- Displays a **mobile-style interactive UI** right inside Jupyter
- Lets you click any tag to see a detailed AI-generated analysis

---
**Setup:** Just add your Anthropic API key in Cell 2, then run all cells (`Kernel → Restart & Run All`)

In [1]:
# ── CELL 1: Install dependencies ─────────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "anthropic", "ipywidgets", "-q"])
print("✅ Dependencies ready")

✅ Dependencies ready


In [2]:
# ── CELL 2: Configuration — ADD YOUR API KEY HERE ─────────────────────────────
ANTHROPIC_API_KEY = "sk-ant-..."   # ← paste your key here

# ─────────────────────────────────────────────────────────────────────────────
import anthropic, json, re
from datetime import datetime
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

TODAY = datetime.now().strftime("%d %B %Y")
NOW   = datetime.now().strftime("%I:%M %p")

# Category → colour + emoji
CATEGORIES = {
    "क्रिकेट":    ("#00C853", "🏏"),
    "खेल":         ("#FF6D00", "⚽"),
    "मनोरंजन":    ("#D500F9", "🎬"),
    "राजनीति":    ("#FF1744", "🏛️"),
    "समाचार":     ("#2979FF", "📰"),
    "त्योहार":    ("#FFAB00", "🪔"),
    "मौसम":       ("#00B0FF", "🌧️"),
    "फ़िल्म":     ("#F50057", "🎥"),
    "टेक":         ("#00E5FF", "💻"),
    "व्यापार":    ("#76FF03", "📈"),
}
DEFAULT_CAT = ("#FF6B35", "🔥")

def cat_style(category):
    return CATEGORIES.get(category, DEFAULT_CAT)

print(f"✅ Config ready | Today: {TODAY} | {NOW}")

✅ Config ready | Today: 03 May 2026 | 06:33 PM


In [3]:
pip install feedparser

Defaulting to user installation because normal site-packages is not writeable
  Using cached feedparser-6.0.12-py3-none-any.whl.metadata (2.7 kB)
  Using cached sgmllib3k-1.0.0.tar.gz (5.8 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
Using cached feedparser-6.0.12-py3-none-any.whl (81 kB)
  Created wheel for sgmllib3k: filename=sgmllib3k-1.0.0-py3-none-any.whl size=6104 sha256=2e940e2137605f27cfa34f33b00b9eee827fead123ecff26eb3e0581f3abf016
  Stored in directory: c:\users\reach\appdata\local\pip\cache\wheels\3d\4d\ef\37cdccc18d6fd7e0dd7817dcdf9146d4d6789c32a227a28134
Successfully built sgmllib3k

   -------------------- ------------------- 1/2 [feedparser]
   -------------------- ------------------- 1/

In [4]:
import feedparser
import pandas as pd
import re
from collections import Counter

def fetch_trending_tags(n=12):
    print("🌐 Fetching trends from News...")

    url = "https://news.google.com/rss?hl=en-IN&gl=IN&ceid=IN:en"
    feed = feedparser.parse(url)

    topics = []

    for entry in feed.entries:
        title = entry.title.lower()

        topic = re.split(r'[-|:]', title)[0]
        topic = re.sub(r'[^a-zA-Z0-9\s]', '', topic).strip()

        if len(topic) > 5:
            topics.append(topic)

    counter = Counter(topics)

    df = pd.DataFrame(counter.items(), columns=['trend', 'score'])
    df = df.sort_values(by='score', ascending=False).head(n)

    # Normalize score
    df['heat'] = (df['score'] / df['score'].max() * 100).astype(int)

    # Format output
    results = []
    for _, row in df.iterrows():
        results.append({
            "tag": "#" + row['trend'].replace(" ", "_"),
            "description": f"{row['trend']} अभी भारत में ट्रेंड कर रहा है",
            "category": "समाचार",
            "heat": int(row['heat']),
            "source": "Google News",
            "posts": f"{int(row['heat'] * 500)}+"
        })

    return results

def fetch_detail_summary(trend):
    """
    Given a trend dict, returns a fresh AI-generated Hindi summary with web search.
    """
    response = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=600,
        system="""You are a sharp Hindi social media writer for ShareChat.
Write in natural conversational Hindi (Devanagari). Keep it punchy and real.
Search the web for the latest info on this topic before writing.""",
        messages=[{
            "role": "user",
            "content": f"""Topic: {trend['tag']}
Category: {trend['category']}
Description: {trend['description']}

Search for latest news on this topic then write a 4-5 sentence Hindi social media summary of:
- What happened / what is happening
- Why Indians are talking about it
- Public reaction
End with 3 related hashtags.
Write ONLY in Hindi Devanagari script."""
        }],
        tools=[{"type": "web_search_20250305", "name": "web_search"}]
    )
    return "".join(
        block.text for block in response.content if hasattr(block, "text")
    ).strip()


print("✅ Functions defined")

✅ Functions defined


In [5]:
# ── CELL 4: HTML rendering helpers ───────────────────────────────────────────

def heat_bar_html(score, color):
    labels = ["", "गरम", "बहुत गरम", "धमाकेदार", "आग लगी!", "तूफ़ान!"]
    level = max(1, min(5, round(score / 20)))
    return f"""
    <div style="display:flex;align-items:center;gap:6px;margin-top:4px">
      <div style="flex:1;height:3px;background:#ffffff15;border-radius:4px;overflow:hidden">
        <div style="width:{score}%;height:100%;background:linear-gradient(90deg,{color},{color}aa);border-radius:4px"></div>
      </div>
      <span style="font-size:9px;color:{color};font-weight:700;letter-spacing:.5px">{labels[level]}</span>
    </div>"""


def trend_card_html(trend, rank):
    color, emoji = cat_style(trend.get("category", ""))
    is_top3 = rank <= 3
    border = f"1px solid {color}40" if is_top3 else "1px solid #ffffff08"
    bg     = f"linear-gradient(135deg,#16161f,#1e1e2e)" if is_top3 else "#13131c"
    fire   = "🔥" if is_top3 else ""
    rank_bg = f"background:linear-gradient(135deg,{color}30,{color}15);color:{color};border:1px solid {color}40" if is_top3 \
              else "background:#ffffff08;color:#ffffff40"
    top_line = f"<div style='position:absolute;top:0;left:0;right:0;height:2px;background:linear-gradient(90deg,transparent,{color},transparent)'></div>" if is_top3 else ""

    return f"""
    <div style="background:{bg};border-radius:16px;padding:14px 16px;margin-bottom:10px;
                border:{border};position:relative;overflow:hidden;cursor:pointer"
         class="trend-card" data-rank="{rank}">
      {top_line}
      <div style="display:flex;align-items:flex-start;gap:12px">
        <div style="min-width:32px;height:32px;border-radius:10px;{rank_bg};
                    display:flex;align-items:center;justify-content:center;
                    font-size:{'14' if is_top3 else '12'}px;font-weight:800;flex-shrink:0">
          {rank}
        </div>
        <div style="flex:1;min-width:0">
          <div style="font-size:15px;font-weight:800;color:#fff;line-height:1.2;margin-bottom:3px">
            {trend['tag']} {fire}
          </div>
          <div style="font-size:12px;color:#ffffff65;line-height:1.4;margin-bottom:6px">
            {trend.get('description','')[:100]}{'…' if len(trend.get('description','')) > 100 else ''}
          </div>
          {heat_bar_html(trend.get('heat', 70), color)}
          <div style="display:flex;align-items:center;gap:6px;margin-top:8px;flex-wrap:wrap">
            <span style="font-size:10px;font-weight:700;color:{color};
                         background:{color}18;padding:2px 8px;border-radius:20px;
                         border:1px solid {color}30">{emoji} {trend.get('category','')}</span>
            <span style="font-size:10px;color:#ffffff40;background:#ffffff08;
                         padding:2px 8px;border-radius:20px">📡 {trend.get('source','')}</span>
            <span style="font-size:10px;color:#ffffff30;margin-left:auto">{trend.get('posts','')} पोस्ट</span>
          </div>
        </div>
        <div style="color:#ffffff20;font-size:18px;flex-shrink:0;margin-top:4px">›</div>
      </div>
    </div>"""


def featured_card_html(trend):
    color, emoji = cat_style(trend.get("category", ""))
    return f"""
    <div style="background:linear-gradient(135deg,{color}20,#16161f);border-radius:20px;
                padding:20px;margin-bottom:14px;border:1px solid {color}40;
                position:relative;overflow:hidden;cursor:pointer"
         class="trend-card" data-rank="1">
      <div style="position:absolute;top:-20px;right:-10px;font-size:80px;opacity:0.06">{emoji}</div>
      <div style="font-size:10px;color:{color};font-weight:800;letter-spacing:2px;margin-bottom:8px">
        🔥 आज का #1 ट्रेंड
      </div>
      <div style="font-size:22px;font-weight:800;color:#fff;margin-bottom:6px">{trend['tag']}</div>
      <div style="font-size:13px;color:#ffffffaa;line-height:1.5;margin-bottom:12px">
        {trend.get('description','')}
      </div>
      <div style="display:flex;align-items:center;gap:8px;flex-wrap:wrap">
        <span style="font-size:11px;color:{color};background:{color}18;padding:3px 10px;
                     border-radius:20px;font-weight:700;border:1px solid {color}40">{emoji} {trend.get('category','')}</span>
        <span style="font-size:11px;color:#ffffff40;background:#ffffff08;
                     padding:3px 10px;border-radius:20px">Heat: {trend.get('heat',0)}/100</span>
        <span style="margin-left:auto;color:#ffffff30;font-size:18px">›</span>
      </div>
    </div>"""


def build_feed_html(trends):
    cards = ""
    if trends:
        cards += featured_card_html(trends[0])
        for i, t in enumerate(trends[1:], start=2):
            cards += trend_card_html(t, i)

    return f"""
    <style>
      @import url('https://fonts.googleapis.com/css2?family=Noto+Sans+Devanagari:wght@400;600;700;800&display=swap');
      .sc-wrap * {{ box-sizing:border-box; font-family:'Noto Sans Devanagari',sans-serif; }}
      @keyframes pulse {{ 0%,100%{{opacity:1}} 50%{{opacity:.4}} }}
    </style>
    <div class="sc-wrap" style="max-width:420px;margin:0 auto;background:#0A0A0F;
                                border-radius:24px;overflow:hidden;box-shadow:0 0 60px #FF6B3520">
      <!-- Status bar -->
      <div style="height:44px;display:flex;align-items:center;justify-content:space-between;
                  padding:0 20px;background:#0A0A0F">
        <span style="font-size:12px;font-weight:600;color:#fff">{NOW}</span>
        <span style="font-size:12px;color:#fff">📶 🔋</span>
      </div>
      <!-- App header -->
      <div style="padding:8px 20px 14px;border-bottom:1px solid #ffffff08;background:#0A0A0F">
        <div style="display:flex;align-items:center;justify-content:space-between">
          <div style="display:flex;align-items:center;gap:10px">
            <div style="width:36px;height:36px;border-radius:10px;
                        background:linear-gradient(135deg,#FF6B35,#FF1744);
                        display:flex;align-items:center;justify-content:center;font-size:18px">🔥</div>
            <div>
              <div style="font-size:18px;font-weight:800;color:#fff;line-height:1">ट्रेंडिंग</div>
              <div style="font-size:10px;color:#ffffff40;margin-top:1px">{NOW} पर अपडेट • {TODAY}</div>
            </div>
          </div>
          <span style="background:#FF6B3520;border:1px solid #FF6B3540;color:#FF6B35;
                       border-radius:10px;padding:6px 12px;font-size:11px;font-weight:700">ShareChat</span>
        </div>
        <!-- Live dot -->
        <div style="display:flex;align-items:center;gap:6px;margin-top:10px">
          <div style="width:8px;height:8px;border-radius:50%;background:#00C853;
                      animation:pulse 1.5s infinite"></div>
          <span style="font-size:11px;color:#00C853;font-weight:600">LIVE — भारत में अभी ट्रेंड हो रहा है</span>
        </div>
      </div>
      <!-- Category pills -->
      <div style="display:flex;gap:8px;padding:12px 20px;overflow-x:auto;background:#0A0A0F">
        {''.join(f'<span style="flex-shrink:0;padding:5px 14px;border-radius:20px;font-size:12px;font-weight:700;background:{"#FF6B35" if i==0 else "#ffffff08"};color:{"#fff" if i==0 else "#ffffff50"};border:{"none" if i==0 else "1px solid #ffffff10"}">{ c }</span>' for i,c in enumerate(["सभी","क्रिकेट","फ़िल्म","राजनीति","खेल","समाचार"]))}
      </div>
      <!-- Cards -->
      <div style="padding:4px 16px 16px;background:#0A0A0F">
        {cards}
      </div>
      <!-- Bottom nav -->
      <div style="display:flex;justify-content:space-around;padding:10px 0 16px;
                  background:#0d0d16;border-top:1px solid #ffffff10">
        {''.join(f'<div style="display:flex;flex-direction:column;align-items:center;gap:3px"><span style="font-size:20px">{ic}</span><span style="font-size:9px;font-weight:{"700" if lbl=="ट्रेंड" else "500"};color:{"#FF6B35" if lbl=="ट्रेंड" else "#ffffff30"}">{lbl}</span></div>' for ic,lbl in [("🏠","होम"),("🔥","ट्रेंड"),("➕","बनाएं"),("🔔","नोटिफ़"),("👤","प्रोफाइल")])}
      </div>
    </div>"""


def build_detail_html(trend, summary):
    color, emoji = cat_style(trend.get("category", ""))
    heat = trend.get('heat', 70)
    return f"""
    <style>
      @import url('https://fonts.googleapis.com/css2?family=Noto+Sans+Devanagari:wght@400;600;700;800&display=swap');
      .sc-det * {{ box-sizing:border-box;font-family:'Noto Sans Devanagari',sans-serif; }}
    </style>
    <div class="sc-det" style="max-width:420px;margin:0 auto;background:#0A0A0F;
                               border-radius:24px;overflow:hidden;box-shadow:0 0 60px {color}20">
      <!-- Header -->
      <div style="padding:24px 20px 16px;background:linear-gradient(180deg,{color}15,transparent)">
        <div style="width:56px;height:56px;border-radius:16px;
                    background:linear-gradient(135deg,{color}30,{color}15);
                    border:1px solid {color}50;display:flex;align-items:center;
                    justify-content:center;font-size:28px;margin-bottom:12px">{emoji}</div>
        <div style="font-size:22px;font-weight:800;color:#fff;margin-bottom:8px;line-height:1.2">{trend['tag']}</div>
        <div style="display:flex;gap:6px;flex-wrap:wrap">
          <span style="font-size:11px;font-weight:700;color:{color};background:{color}18;
                       padding:3px 10px;border-radius:20px;border:1px solid {color}40">{emoji} {trend.get('category','')}</span>
          <span style="font-size:11px;color:#ffffff50;background:#ffffff0a;
                       padding:3px 10px;border-radius:20px">Heat: {heat}/100</span>
          <span style="font-size:11px;color:#ffffff50;background:#ffffff0a;
                       padding:3px 10px;border-radius:20px">📡 {trend.get('source','')}</span>
        </div>
      </div>
      <!-- Heat bar -->
      <div style="padding:0 20px 16px">
        <div style="height:4px;background:#ffffff0a;border-radius:4px;overflow:hidden">
          <div style="width:{heat}%;height:100%;background:linear-gradient(90deg,{color},#FF1744);border-radius:4px"></div>
        </div>
      </div>
      <!-- Description -->
      <div style="padding:0 20px 16px">
        <div style="background:#16161f;border-radius:16px;padding:16px;border:1px solid #ffffff08">
          <div style="font-size:11px;color:#ffffff40;margin-bottom:6px;letter-spacing:1px">विवरण</div>
          <div style="font-size:14px;color:#ffffffcc;line-height:1.6">{trend.get('description','')}</div>
        </div>
      </div>
      <!-- AI Summary -->
      <div style="padding:0 20px 16px">
        <div style="background:linear-gradient(135deg,{color}08,#16161f);border-radius:16px;
                    padding:16px;border:1px solid {color}20">
          <div style="display:flex;align-items:center;gap:6px;margin-bottom:10px">
            <span style="font-size:14px">✨</span>
            <span style="font-size:11px;color:{color};font-weight:700;letter-spacing:1px">AI विश्लेषण</span>
          </div>
          <div style="font-size:14px;color:#ffffffcc;line-height:1.7;white-space:pre-wrap">{summary}</div>
        </div>
      </div>
      <!-- Stats -->
      <div style="padding:0 20px 16px;display:grid;grid-template-columns:1fr 1fr;gap:10px">
        <div style="background:#16161f;border-radius:14px;padding:14px;border:1px solid #ffffff08;text-align:center">
          <div style="font-size:22px;margin-bottom:4px">📝</div>
          <div style="font-size:16px;font-weight:800;color:#fff">{trend.get('posts','N/A')}</div>
          <div style="font-size:11px;color:#ffffff40;margin-top:2px">पोस्ट</div>
        </div>
        <div style="background:#16161f;border-radius:14px;padding:14px;border:1px solid #ffffff08;text-align:center">
          <div style="font-size:22px;margin-bottom:4px">🔥</div>
          <div style="font-size:16px;font-weight:800;color:#fff">{heat}/100</div>
          <div style="font-size:11px;color:#ffffff40;margin-top:2px">हीट स्कोर</div>
        </div>
      </div>
      <div style="padding:0 20px 24px">
        <div style="width:100%;padding:14px;border-radius:14px;
                    background:linear-gradient(135deg,{color},{color}cc);
                    color:#fff;font-size:15px;font-weight:700;text-align:center">
          इस ट्रेंड को एक्सप्लोर करें →
        </div>
      </div>
    </div>"""


print("✅ HTML rendering helpers ready")

✅ HTML rendering helpers ready


In [6]:
# ── CELL 5: Fetch trends (run this cell to refresh) ───────────────────────────
TRENDS = fetch_trending_tags(n=12)

# Quick preview table
print(f"\n{'Rank':<5} {'Tag':<30} {'Category':<15} {'Heat':>5} {'Posts':>8}")
print("-" * 68)
for i, t in enumerate(TRENDS, 1):
    print(f"{i:<5} {t.get('tag',''):<30} {t.get('category',''):<15} {t.get('heat',0):>5} {t.get('posts',''):>8}")

🌐 Fetching trends from News...

Rank  Tag                            Category         Heat    Posts
--------------------------------------------------------------------
1     #10_lifetimes_wont_be_enough   समाचार            100   50000+
2     #ipl_2026                      समाचार            100   50000+
3     #todays_nyt_strands_hints_and_answer_for_sunday_may_3_2026 समाचार            100   50000+
4     #diljit_dosanjh_roars_jinne_jhande_dikhane_dilkhai_challo_at_protesters_waving_pro समाचार            100   50000+
5     #from_firing_near_kapil_sharmas_kaps_cafe_to_bishnoi_gangs_warning समाचार            100   50000+
6     #deepika_padukones_23k_mes_demoiselles_dress_goes_viral_on_king_sets_with_shah_rukh_khan समाचार            100   50000+
7     #sundar_c_on_quitting_rajinikanth समाचार            100   50000+
8     #srh_vs_kkr_live_score_ipl_2026 समाचार            100   50000+
9     #mumbai_indians_not_ready_to_drop_suryakumar_yadav_but_recurring_ipl_2026_struggles_raise_red_flags समा

In [7]:
# ── CELL 6: Interactive Feed UI ───────────────────────────────────────────────
#
# Renders the mobile ShareChat-style feed.
# Click any numbered button below the feed to see the detail view for that trend.

output_area = widgets.Output()

def show_feed():
    with output_area:
        clear_output(wait=True)
        display(HTML(build_feed_html(TRENDS)))

def show_detail(rank):
    trend = TRENDS[rank - 1]
    with output_area:
        clear_output(wait=True)
        display(HTML("<p style='font-family:sans-serif;color:#888'>✨ AI विश्लेषण लोड हो रहा है…</p>"))
    summary = fetch_detail_summary(trend)
    with output_area:
        clear_output(wait=True)
        display(HTML(build_detail_html(trend, summary)))

# ── Controls ──────────────────────────────────────────────────────────────────
back_btn = widgets.Button(
    description="← वापस जाएं (Feed)",
    button_style="",
    style={"button_color": "#1e1e2e", "font_weight": "bold"},
    layout=widgets.Layout(width="180px", margin="4px")
)
back_btn.on_click(lambda b: show_feed())

detail_buttons = []
for i, t in enumerate(TRENDS, 1):
    color, emoji = cat_style(t.get("category", ""))
    btn = widgets.Button(
        description=f"{i}. {t['tag'][:22]}",
        button_style="",
        style={"button_color": "#16161f", "font_weight": "600"},
        layout=widgets.Layout(width="220px", margin="3px")
    )
    btn.rank = i
    btn.on_click(lambda b: show_detail(b.rank))
    detail_buttons.append(btn)

# Layout: left = controls, right = output
controls = widgets.VBox(
    [widgets.HTML("<b style='font-family:sans-serif;font-size:13px;color:#aaa'>👆 किसी ट्रेंड पर क्लिक करें:</b>"),
     back_btn,
     widgets.HTML("<hr style='border-color:#333;margin:6px 0'>")]
    + detail_buttons,
    layout=widgets.Layout(width="240px", padding="8px",
                          border="1px solid #333", border_radius="12px")
)

ui = widgets.HBox([controls, output_area],
                  layout=widgets.Layout(gap="16px", align_items="flex-start"))

show_feed()     # render feed first
display(ui)

In [8]:
# ── CELL 7: Export — save trending tags to JSON ───────────────────────────────
import json, os

export = {
    "generated_at": datetime.now().isoformat(),
    "date": TODAY,
    "total_tags": len(TRENDS),
    "tags": TRENDS
}

with open("trending_tags_export.json", "w", encoding="utf-8") as f:
    json.dump(export, f, ensure_ascii=False, indent=2)

print(f"✅ Exported to trending_tags_export.json")
print(f"\n📋 Sample output (first 3 tags):")
for t in TRENDS[:3]:
    print(f"  {t.get('tag')} | {t.get('category')} | Heat: {t.get('heat')} | {t.get('description','')[:60]}...")

✅ Exported to trending_tags_export.json

📋 Sample output (first 3 tags):
  #10_lifetimes_wont_be_enough | समाचार | Heat: 100 | 10 lifetimes wont be enough अभी भारत में ट्रेंड कर रहा है...
  #ipl_2026 | समाचार | Heat: 100 | ipl 2026 अभी भारत में ट्रेंड कर रहा है...
  #todays_nyt_strands_hints_and_answer_for_sunday_may_3_2026 | समाचार | Heat: 100 | todays nyt strands hints and answer for sunday may 3 2026 अभ...


In [9]:
pip install deep_translator

Defaulting to user installation because normal site-packages is not writeable
  Using cached deep_translator-1.11.4-py3-none-any.whl.metadata (30 kB)
Using cached deep_translator-1.11.4-py3-none-any.whl (42 kB)
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [10]:
from deep_translator import GoogleTranslator

def to_hindi(text):
    try:
        return GoogleTranslator(source='auto', target='hi').translate(text)
    except:
        return text

In [11]:
import pandas as pd

rows = []

for t in TRENDS:
    hindi_text = to_hindi(t["tag"].replace("#", "").replace("_", " "))

    rows.append({
        "tag": "#" + hindi_text.replace(" ", "_"),   # Hindi hashtag
        "description": to_hindi(t["description"]),
        "category": to_hindi(t["category"]),
        "score": t["heat"],
        "source": t["source"],
        "posts": t["posts"]
    })

df = pd.DataFrame(rows)

df.to_csv("trending_tags_output.csv", index=False, encoding="utf-8-sig")

print("✅ Hindi CSV exported")

✅ Hindi CSV exported


In [12]:
# ── CELL 8: System design diagram (text) ─────────────────────────────────────
diagram = """
╔══════════════════════════════════════════════════════════════════╗
║          SHARECHAT TRENDING TAGS — PIPELINE DIAGRAM              ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  ┌─────────────┐    ┌──────────────────┐    ┌────────────────┐  ║
║  │  Web Sources │───▶│  Claude Sonnet   │───▶│  JSON Parser   │  ║
║  │  (Live Search│    │  + web_search    │    │  + Validator   │  ║
║  │   via Claude)│    │  tool enabled    │    │                │  ║
║  └─────────────┘    └──────────────────┘    └───────┬────────┘  ║
║                                                      │           ║
║  Sources searched:                                   ▼           ║
║  • News (NDTV, AajTak,       ┌──────────────────────────────┐   ║
║    TimesNow, India Today)     │  Ranked Tag List (by heat)   │   ║
║  • Social signals (Twitter/  │  tag / description / category │   ║
║    X viral, YouTube trends)  │  heat / source / posts       │   ║
║  • IPL / Sports live scores  └──────────────┬───────────────┘   ║
║  • Bollywood / OTT releases                  │                   ║
║  • Festival calendar                         ▼                   ║
║                              ┌──────────────────────────────┐   ║
║                              │   Feed UI (ipywidgets HTML)  │   ║
║                              │   Mobile-style dark theme    │   ║
║                              └──────────────┬───────────────┘   ║
║                                             │  User taps tag    ║
║                                             ▼                   ║
║                              ┌──────────────────────────────┐   ║
║                              │  Detail View                 │   ║
║                              │  Claude (2nd call + search)  │   ║
║                              │  → Hindi AI summary          │   ║
║                              └──────────────────────────────┘   ║
║                                                                  ║
╠══════════════════════════════════════════════════════════════════╣
║  HEAT SCORE LOGIC:                                               ║
║  • Recency boost  (+20 if breaking in last 2 hrs)                ║
║  • Search volume  (estimated by Claude from news density)        ║
║  • Social velocity (mentions & shares signal)                    ║
║  • Category weight (cricket/IPL gets +10 in India)               ║
║  • Festival calendar match (+15 if current festival)             ║
╚══════════════════════════════════════════════════════════════════╝
"""
print(diagram)


╔══════════════════════════════════════════════════════════════════╗
║          SHARECHAT TRENDING TAGS — PIPELINE DIAGRAM              ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  ┌─────────────┐    ┌──────────────────┐    ┌────────────────┐  ║
║  │  Web Sources │───▶│  Claude Sonnet   │───▶│  JSON Parser   │  ║
║  │  (Live Search│    │  + web_search    │    │  + Validator   │  ║
║  │   via Claude)│    │  tool enabled    │    │                │  ║
║  └─────────────┘    └──────────────────┘    └───────┬────────┘  ║
║                                                      │           ║
║  Sources searched:                                   ▼           ║
║  • News (NDTV, AajTak,       ┌──────────────────────────────┐   ║
║    TimesNow, India Today)     │  Ranked Tag List (by heat)   │   ║
║  • Social signals (Twitter/  │  tag / description / category │   ║
║    X viral, YouTube trends)  │  he

In [13]:
diagram="""         ┌────────────────────┐
                    │   External Signals │
                    │                    │
                    │  Google News RSS   │
                    │  (Real-time feed)  │
                    └─────────┬──────────┘
                              ↓
                    ┌────────────────────┐
                    │   Ingestion Layer  │
                    │  • Fetch RSS       │
                    │  • Parse XML       │
                    └─────────┬──────────┘
                              ↓
                    ┌────────────────────┐
                    │ Signal Processing  │
                    │                    │
                    │ • Headline parsing │
                    │ • Topic extraction │
                    │ • Token cleaning   │
                    └─────────┬──────────┘
                              ↓
                    ┌────────────────────┐
                    │ Trend Aggregation  │
                    │                    │
                    │ • Frequency count  │
                    │ • Merge duplicates │
                    └─────────┬──────────┘
                              ↓
                    ┌────────────────────┐
                    │ Scoring Engine     │
                    │                    │
                    │ Score = f(         │
                    │  frequency,        │
                    │  recency proxy     │
                    │ )                  │
                    └─────────┬──────────┘
                              ↓
                    ┌────────────────────┐
                    │ Ranking Layer      │
                    │ • Top N selection  │
                    └─────────┬──────────┘
                              ↓
                    ┌────────────────────┐
                    │ Enrichment Layer   │
                    │                    │
                    │ • Hindi translation│
                    │ • Category tagging │
                    │ • Description gen  │
                    └─────────┬──────────┘
                              ↓
                    ┌────────────────────┐
                    │ Delivery Layer     │
                    │                    │
                    │ • CSV output       │
                    │ • Streamlit app    │
                    └────────────────────┘"""

SyntaxError: invalid character '┌' (U+250C) (167574822.py, line 1)